# Qwen2.5-1.5B Reasoning v2 — Colab GPU 训练流程

**适用场景**：本地无 GPU，已在本地用 `run_local.sh` 完成数据准备 / API 评测 / Teacher 数据生成。

**Colab 运行环境**：A100 (40GB) 推荐 / L4 / T4 也可（T4 自动切 fp16）。

**流程**：安装依赖 → 挂载 Drive → 同步本地产物 → 克隆代码 → 跑 `run_gpu.sh` → 验证。

> 断线重连：从 **Cell 3（挂载 Drive）** 开始即可，Drive 里的产物不会丢失。

In [ ]:
# Cell 1：安装依赖（每次新 Session 都要跑）
!pip install -q unsloth trl peft bitsandbytes transformers datasets accelerate pyyaml safetensors tqdm

import unsloth, trl, transformers, peft
print(f"✅ Unsloth {unsloth.__version__} | TRL {trl.__version__} | Transformers {transformers.__version__} | PEFT {peft.__version__}")

In [ ]:
# Cell 2：从 Colab Secrets 加载 HF_TOKEN 和 DASHSCOPE_API_KEY
# 在 Colab 左侧 🔑 Secrets 面板里添加这两个 key
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
try:
    os.environ["DASHSCOPE_API_KEY"] = userdata.get("DASHSCOPE_API_KEY")
    print("✅ HF_TOKEN + DASHSCOPE_API_KEY 已加载")
except Exception:
    print("⚠️ 未配置 DASHSCOPE_API_KEY，Iterative DPO 在线模式将不可用（但基础训练 OK）")

In [ ]:
# Cell 3：挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!mkdir -p "{DRIVE_DIR}/data/processed" "{DRIVE_DIR}/outputs" "{DRIVE_DIR}/logs"
print(f"✅ Drive 已挂载，工作根目录：{DRIVE_DIR}")

In [ ]:
# Cell 4：克隆 / 更新代码（数据缓存放在 Drive，不受影响）
import os

PROJECT_DIR = "/content/6000Q-QwenMiniReason"
REPO_URL    = "https://github.com/yukiiii0730/6000Q-QwenMiniReason.git"  # ← 改成你的仓库

if not os.path.exists(PROJECT_DIR):
    print("📥 首次克隆项目...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("🔄 项目已存在，拉取最新代码...")
    !git -C {PROJECT_DIR} fetch origin && git -C {PROJECT_DIR} reset --hard origin/main

%cd {PROJECT_DIR}
!git log --oneline -3

In [ ]:
# Cell 5：从 Drive 同步本地阶段的产物到项目目录（data + logs）
# 本地运行 run_local.sh 后，请把 data/processed/ 和 logs/ 上传到 Drive 的 Qwen-Reasoning/ 下
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"

!mkdir -p data/processed logs
!rsync -av --info=progress2 "{DRIVE_DIR}/data/processed/" data/processed/ || true
!rsync -av --info=progress2 "{DRIVE_DIR}/logs/" logs/ || true

print("\n📦 已同步的本地产物：")
!ls -la data/processed/ 2>/dev/null | head -20
!echo '---'
!ls -la logs/ 2>/dev/null | head -20

In [ ]:
# Cell 6：检测 GPU + 把 outputs 重定向到 Drive（防断线丢失）
import torch, yaml

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb  = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
print(f"GPU : {gpu_name}")
print(f"显存: {vram_gb} GB")

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"

for cfg_path, out_key, base_key in [
    ("config/sft_config.yaml", "output_dir", None),
    ("config/dpo_config.yaml", "output_dir", "base_adapter_path"),
]:
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    cfg[out_key] = f"{DRIVE_DIR}/outputs/" + ("sft" if "sft" in cfg_path else "dpo")
    if base_key:
        cfg[base_key] = f"{DRIVE_DIR}/outputs/sft"
    # T4 自动切 fp16
    if "T4" in gpu_name or not torch.cuda.is_bf16_supported():
        cfg["train"]["bf16"] = False
        cfg["train"]["fp16"] = True
        for st in cfg.get("stages", []):
            if st.get("train"):
                st["train"]["bf16"] = False
                st["train"]["fp16"] = True
    # 显存 < 20GB 进一步降采样
    if vram_gb < 20:
        for st in cfg.get("stages", []):
            ds = st.get("dataset", {})
            if ds.get("max_samples", 0) > 8000:
                ds["max_samples"] = 8000
    with open(cfg_path, "w") as f:
        yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print("✅ 配置已自动适配，outputs 重定向到 Drive")

In [ ]:
# Cell 7：一键运行 GPU 阶段（SFT 课程 → DPO → 评测）
# 推荐参数：使用本地预过滤的高质量数据 + Teacher-Guided DPO 数据
!chmod +x run_gpu.sh && bash run_gpu.sh --use-filtered-data --use-teacher-dpo

**Cell 7 备选用法**：

```bash
!bash run_gpu.sh                          # 标准流程（不使用本地过滤数据）
!bash run_gpu.sh --quick                  # 快速测试（SFT=50 / DPO=30 步）
!bash run_gpu.sh --skip-sft               # 跳过 SFT 直接做 DPO
!bash run_gpu.sh --iterative-dpo 2        # 额外 2 轮 Iterative DPO（在线模式调 Teacher API）
!bash run_gpu.sh --use-filtered-data --use-teacher-dpo --iterative-dpo 2  # 全量优化
```

In [ ]:
# Cell 8：训练完成后的对比与可视化
!python eval/compare_table.py 2>/dev/null || echo '(compare_table 找不到 result，先确保上一步训练已完成)'

import json, os
summary = {}
for tag in ["baseline", "qwen25_7b", "qwen25_14b", "qwen3_235b", "sft", "result"]:
    for ds in ["gsm8k", "bbh"]:
        p = f"logs/{ds}_{tag}.json"
        if os.path.exists(p):
            d = json.load(open(p))
            summary[f"{ds}_{tag}"] = round(d.get("accuracy", 0), 4)
print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
# Cell 9：把训练产物 / 评测结果同步回 Drive（防断线丢失，方便本地拉回）
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!rsync -av --info=progress2 outputs/  "{DRIVE_DIR}/outputs/"
!rsync -av --info=progress2 logs/     "{DRIVE_DIR}/logs/"
print("✅ 已同步到 Drive，本地用 `rclone` 或 Drive 网页下载即可")

In [ ]:
# Cell 10（可选）：推理对比 — 基础模型 vs 微调模型
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

DRIVE_DIR  = "/content/drive/MyDrive/Qwen-Reasoning"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
FT_MODEL   = f"{DRIVE_DIR}/outputs/merged"
QUESTION   = "小明有 12 个苹果，给了小红 3 个，又买了 5 个，现在有几个？请一步步推理后给出答案。"

def infer(model_path, prompt, max_new=512):
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_path, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
    )
    inputs  = tok(prompt, return_tensors="pt").to(mdl.device)
    outputs = mdl.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    return tok.decode(outputs[0], skip_special_tokens=True)

print("=" * 60)
print("【基础模型】")
print(infer(BASE_MODEL, QUESTION))
print("\n" + "=" * 60)
print("【微调模型（SFT + DPO）】")
print(infer(FT_MODEL, QUESTION))